In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
with open(PROJECT_ROOT / 'config' / 'experiment.yaml') as f:
    cfg = yaml.safe_load(f)

from src.data import DataLoader
from src.methods import FraudDetector
from src.methods.si import SynapticIntelligence, train_with_si
from src.utils import test_on_task, set_seed, compute_thesis_metrics

# ── Hardcoded config ──────────────────────────────────────────────────────────
DATASET   = "axis1_delta1.5_sudden"
SI_LAMBDA = 100      # hardcoded for this test
SI_XI     = 0.01
N_TASKS   = 10
N_EPOCHS  = 20
LR        = 0.01
SEED      = 42

set_seed(SEED)

# ── Load dataset ──────────────────────────────────────────────────────────────
csv_path = PROJECT_ROOT / cfg['paths']['base_dir'] / f"{DATASET}.csv"
loader = DataLoader(csv_path)

# ── Train SI ──────────────────────────────────────────────────────────────────
model = FraudDetector(input_features=8)
si    = SynapticIntelligence(xi=SI_XI)

acc_mat = np.zeros((N_TASKS, N_TASKS))
pr_mat  = np.zeros((N_TASKS, N_TASKS))

for task_id in range(N_TASKS):
    X_train, y_train = loader.get_data_for_task(task_id, "train")
    model, small_omega, prev_params = train_with_si(
        model, X_train, y_train, si,
        importance_factor=SI_LAMBDA,
        epochs=N_EPOCHS,
        lr=LR,
    )
    si.register_task(model, small_omega, prev_params)

    for test_id in range(task_id + 1):
        X_test, y_test = loader.get_data_for_task(test_id, "test")
        m = test_on_task(model, X_test, y_test)
        acc_mat[task_id, test_id] = m["acc"]
        pr_mat[task_id, test_id]  = m["pr_auc"]

    print(f"Task {task_id + 1}/{N_TASKS} done  |  diag PR-AUC = {pr_mat[task_id, task_id]:.3f}")

# ── Metrics ───────────────────────────────────────────────────────────────────
_, _, avg_pr, forget_pr, diag_pr = compute_thesis_metrics(acc_mat, pr_mat)
print(f"\nAvgPR    = {avg_pr:.4f}")
print(f"ForgetPR = {forget_pr:.4f}")
print(f"DiagPR   = {diag_pr:.4f}")

# ── Plot PR-AUC heatmap ───────────────────────────────────────────────────────
n = N_TASKS
task_labels = [f"T{t}" for t in range(1, n + 1)]
masked = np.where(pr_mat == 0, np.nan, pr_mat)

fig, ax = plt.subplots(figsize=(10, 9))
fig.suptitle(
    f"PR-AUC Heatmap — {DATASET}  (λ={SI_LAMBDA})\n"
    "Row = after training on task | Column = tested on task | Diagonal = peak performance",
    fontsize=13, fontweight="bold",
)
ax.set_title("SI", fontsize=13, fontweight="bold", color="#8172B2", pad=12)

im = ax.imshow(masked, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")

for row in range(n):
    for col in range(n):
        val = pr_mat[row, col]
        if val > 0:
            ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color="black", fontweight="bold")

for t in range(n):
    ax.add_patch(plt.Rectangle(
        (t - 0.5, t - 0.5), 1, 1,
        fill=False, edgecolor="steelblue", linewidth=2.5,
    ))

ax.set_xlabel("Test Task", fontsize=11)
ax.set_ylabel("After Training On Task", fontsize=11)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(task_labels, fontsize=9)
ax.set_yticklabels(task_labels, fontsize=9)
plt.colorbar(im, ax=ax, label="PR-AUC", fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()